# Fast-VTON — Stage 1 trên Colab A100

Notebook này **chỉ điều phối**: mọi logic nằm trong `src/`, ở đây chỉ gọi.
Đọc `docs/VTON_PLAN.md` để hiểu vì sao mỗi bước lại như vậy.

**Luồng chạy**

| Bước | Việc | Thời gian |
|---|---|---|
| 1–3 | Môi trường, repo, checkpoint | ~25 phút |
| 4 | Dữ liệu VITON-HD + **cổng chặn: kiểm tra mask** | ~20 phút |
| 5 | Embedding rỗng cho F_θ | ~2 phút |
| 6 | **Cổng chặn: overfit 8 mẫu** | ~10 phút |
| 7 | Cache đầy đủ 11,647 mẫu | ~30 phút |
| 8 | Train Stage 1 | nhiều giờ |
| 9 | Đóng gói 1 file → Google Drive | ~5 phút |

Hai cổng chặn có dấu ⛔ — sai ở đó thì mọi thứ phía sau vô nghĩa, **đừng chạy tiếp**.

> **Runtime:** Runtime → Change runtime type → **A100 GPU**.
> Colab Pro thường cấp A100 **40 GB**; bản 80 GB không đảm bảo. Cell 1 sẽ tự dò và
> chọn batch size phù hợp.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Cài môi trường và lấy code

Mặc định clone từ `REPO_URL` — **nhớ push commit mới nhất lên GitHub trước khi chạy**,
Colab lấy đúng những gì có trên remote.

Muốn chạy code chưa push thì đặt `USE_DRIVE_COPY = True` và copy thư mục repo lên Drive
tại đúng `DRIVE_REPO_PATH`.

In [ ]:
# Mount Google Drive to access the dataset and save checkpoints
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
REPO_URL = "https://github.com/hoangtung386/Fast-VTON.git"
USE_DRIVE_COPY = False                                 # True = copy từ Drive thay vì clone
DRIVE_REPO_PATH = "/content/drive/MyDrive/Fast-VTON"   # chỉ dùng khi USE_DRIVE_COPY = True

In [ ]:
import shutil
from pathlib import Path

PROJECT = Path("/content/Fast-VTON")

# copytree ném FileNotFoundError trần trụi nếu nguồn không có — báo rõ nguyên nhân trước.
if USE_DRIVE_COPY and not Path(DRIVE_REPO_PATH).is_dir():
    raise FileNotFoundError(
        f"không thấy {DRIVE_REPO_PATH} trên Drive.\n"
        f"Đặt USE_DRIVE_COPY = False để clone từ {REPO_URL}, "
        "hoặc sửa DRIVE_REPO_PATH cho đúng thư mục thật."
    )

if PROJECT.exists():
    shutil.rmtree(PROJECT)

if USE_DRIVE_COPY:
    shutil.copytree(DRIVE_REPO_PATH, PROJECT)
else:
    !git clone -q {REPO_URL} {PROJECT}

# `!git clone` không làm cell fail khi lỗi; không bắt ở đây thì %cd bên dưới báo lỗi khó hiểu.
if not (PROJECT / "src").is_dir():
    raise RuntimeError(f"lấy code thất bại: không thấy {PROJECT}/src")

%cd {PROJECT}
!ls src

In [ ]:
# Pin đúng stack đã kiểm chứng. Torch trước, phần còn lại sau, numpy cuối cùng.
!pip install -q torch==2.2.1 torchvision==0.17.1
!pip install -q -e '.[vton]'
!pip install -q numpy==1.26.4

print("\nKhởi động lại runtime NẾU Colab báo cần, rồi chạy tiếp từ cell dưới.")

In [ ]:
%cd /content/Fast-VTON
import src, torch

print("Fast-VTON", src.__version__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 3. Tải checkpoint từ Google Drive (`weights.zip`, ~9 GB)

Toàn bộ thư mục `weights/` đã được nén thành `weights.zip` và share public trên Drive.

> File >100 MB nên Drive chèn một trang xác nhận virus-scan: `wget` thẳng sẽ tải về
> **HTML** chứ không phải zip. `gdown` tự xử lý confirm token đó, nên dùng nó.

Giải nén **phải** ra `weights/` nằm ngay trong repo: `CheckpointConfig.root` mặc định là
đường dẫn tương đối `weights` (xem `DEFAULT_WEIGHTS_ROOT` trong `src/constants.py`), được
giải so với thư mục làm việc hiện tại.

> `stabilityai/stable-diffusion-2-1-base` đã bị gỡ khỏi Hub (404 kể cả có token).
> Code dùng mirror `Manojb/stable-diffusion-2-1-base`, đã verify sha256 khớp bản gốc.

In [ ]:
# https://drive.google.com/file/d/1Ruk0gMs16WyZMP2UlTtz50H96ico5_ko/view
WEIGHTS_FILE_ID = "1Ruk0gMs16WyZMP2UlTtz50H96ico5_ko"

!pip install -q --upgrade gdown
!gdown "https://drive.google.com/uc?id={WEIGHTS_FILE_ID}" -O /content/weights.zip
!ls -lh /content/weights.zip

In [ ]:
import shutil
import zipfile
from pathlib import Path

PROJECT = Path("/content/Fast-VTON")
ARCHIVE = Path("/content/weights.zip")
STAGING = Path("/content/_weights_staging")
TARGET = PROJECT / "weights"

# gdown trả về HTML thay vì zip khi link hết hạn share — bắt ngay tại đây.
if not zipfile.is_zipfile(ARCHIVE):
    raise RuntimeError(f"{ARCHIVE} không phải zip hợp lệ — kiểm tra quyền share của file Drive")

shutil.rmtree(STAGING, ignore_errors=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    archive.extractall(STAGING)

# Rác macOS đi kèm khi nén từ Finder.
shutil.rmtree(STAGING / "__MACOSX", ignore_errors=True)
for junk in [*STAGING.rglob("._*"), *STAGING.rglob(".DS_Store")]:
    junk.unlink()

# Zip có thể bọc sẵn thư mục weights/ hoặc đổ thẳng nội dung ra gốc.
entries = list(STAGING.iterdir())
extracted_root = entries[0] if len(entries) == 1 and entries[0].is_dir() else STAGING

shutil.rmtree(TARGET, ignore_errors=True)
shutil.move(str(extracted_root), str(TARGET))
shutil.rmtree(STAGING, ignore_errors=True)
ARCHIVE.unlink()          # ~9 GB, không cần giữ sau khi giải nén

print("weights/:", sorted(p.name for p in TARGET.iterdir()))

In [ ]:
%cd /content/Fast-VTON
from src.vton import CheckpointConfig

CHECKPOINTS = CheckpointConfig()
CHECKPOINTS.validate()          # ném FileNotFoundError nếu thiếu bất cứ thứ gì
print("checkpoint OK:", CHECKPOINTS.root.resolve())

In [ ]:
# Xác nhận checkpoint nguyên vẹn: 686/686 tensor của G phải bit-identical.
!python scripts/dissect_checkpoints.py --compare-generator

## 4. Cổng chặn 1 — dữ liệu VITON-HD và kiểm tra mask

`forgeml/viton_hd` là 11,647 cặp train chính thức, có sẵn cột `agnostic` nên không cần
chạy human parsing. Nhưng **không có `parse`**, nên mask dựng bằng hiệu ảnh
(`build_agnostic_mask`) — cách này không phụ thuộc bảng label LIP/CIHP vốn hay sai.

**Nhìn kỹ lưới ảnh bên dưới.** Mask (hàng 3) phải phủ thân áo + hai cánh tay,
**không** lấn xuống quần, **không** ăn vào mặt. Coverage lành mạnh: 0.10 – 0.35.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("forgeml/viton_hd", split="train")
print(dataset)

In [ ]:
import matplotlib.pyplot as plt

from src.vton import DataConfig, build_agnostic_mask, mask_coverage

DATA = DataConfig()          # 512x384, ngưỡng 12, kernel 9
print(f"latent {DATA.latent_height}x{DATA.latent_width}, pil size {DATA.pil_size}")

fig, axes = plt.subplots(3, 6, figsize=(16, 9))
for column in range(6):
    sample = dataset[column * 1900]
    mask = build_agnostic_mask(
        sample["image"], sample["agnostic"], DATA.pil_size,
        DATA.mask_diff_threshold, DATA.mask_morph_kernel,
    )
    axes[0, column].imshow(sample["image"].resize(DATA.pil_size))
    axes[1, column].imshow(sample["agnostic"].resize(DATA.pil_size))
    axes[2, column].imshow(mask, cmap="gray")
    axes[0, column].set_title(f"coverage {mask_coverage(mask):.3f}", fontsize=9)
    for row in range(3):
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra định lượng trên 200 mẫu trước khi cache cả 11k.
import numpy as np

coverages = [
    mask_coverage(build_agnostic_mask(
        dataset[i]["image"], dataset[i]["agnostic"], DATA.pil_size,
        DATA.mask_diff_threshold, DATA.mask_morph_kernel))
    for i in range(0, 2000, 10)
]
coverages = np.array(coverages)
outliers = ((coverages < 0.05) | (coverages > 0.50)).mean()

print(f"coverage: trung vị {np.median(coverages):.3f}, "
      f"khoảng {coverages.min():.3f}–{coverages.max():.3f}")
print(f"tỉ lệ bất thường: {outliers:.1%}")
assert outliers < 0.05, (
    "Quá nhiều mask bất thường. Chỉnh mask_diff_threshold (lấn thì tăng) "
    "hoặc mask_morph_kernel (thủng thì tăng) trong DataConfig rồi chạy lại."
)
print("\nCỔNG CHẶN 1 ĐẠT")

## 5. Embedding rỗng cho F_θ

Try-on giao nhánh prompt cho quần áo, nên text encoder chỉ còn một việc: sinh hằng số này.

> Phải lấy từ **`stabilityai/sd-turbo`** — đó là text encoder mà `InverseModel` được
> huấn luyện cùng. Lấy nhầm từ SD 2.1-base sẽ ra tensor trông hợp lệ nhưng làm giảm
> chất lượng inversion một cách âm thầm. Script đã tự cảnh báo nếu bạn đổi `--model`.

In [ ]:
!python scripts/make_null_embedding.py --output outputs/null_embedding.pt

import torch

print("shape:", tuple(torch.load("outputs/null_embedding.pt").shape))

## 6. Cổng chặn 2 — overfit 8 mẫu

Cache 8 mẫu rồi train 400 step. Loss **phải** tụt gần về 0. Không tụt nghĩa là có bug
trong đường ống (mask sai chỗ, token nối nhầm, tham số không nhận gradient) —
**không phải** thiếu dữ liệu. Rẻ hơn nhiều so với phát hiện sau 6 giờ A100.

In [ ]:
!python scripts/build_vton_cache.py \
    --limit 8 --batch-size 4 --output outputs/smoke_cache

In [ ]:
!python scripts/train_vton_stage1.py \
    --cache outputs/smoke_cache \
    --batch-size 4 --gradient-accumulation-steps 1 \
    --max-steps 400 --log-every 25 \
    --num-workers 0 \
    --output-dir outputs/smoke_run

Đọc dòng log cuối. Cần thấy:

- `trainable parameters: ~33 M` — nếu ra hàng trăm M là đã lỡ mở đóng băng G.
- `loss` giảm đều và về gần 0.

Nếu loss đứng yên, dừng lại và soát: token có tách đúng ở vị trí 257 không, mask latent
có phải nhị phân không, `conv_in` đã mở lên 9 kênh chưa.

In [ ]:
from pathlib import Path

import torch

ckpt = torch.load(max(Path("outputs/smoke_run").glob("*.pt"), key=lambda p: p.stat().st_mtime), map_location="cpu")
trainable = sum(t.numel() for group in ckpt["weights"].values() for t in group.values())
print(f"step {ckpt['step']}, tham số trainable {trainable / 1e6:.2f} M")
assert 25e6 < trainable < 60e6, "số tham số trainable không đúng kỳ vọng ~33 M"
print("\nCỔNG CHẶN 2 ĐẠT (nếu loss cũng đã hội tụ)")

## 7. Cache đầy đủ

Mọi module Stage 1 đóng băng — F_θ, VAE, DINOv2, CLIP — đều nhận input cố định, nên chạy
một lần rồi cache. Vòng lặp train sau đó bỏ hẳn được một lượt UNet và ~1.7 GB VRAM.

Cache ghi ra thư mục `.npy` memmap: file DINOv2 6 GB không cần nằm vừa RAM.

In [ ]:
!python scripts/build_vton_cache.py \
    --output outputs/vton_cache --batch-size 8

!du -sh outputs/vton_cache && ls -la outputs/vton_cache

In [ ]:
from src.vton import CachedVtonDataset

cache = CachedVtonDataset("outputs/vton_cache")
print(f"{len(cache)} mẫu")
for name, tensor in cache[0].items():
    print(f"  {name:20s} {tuple(tensor.shape)}  {tensor.dtype}")

## 8. Train Stage 1

Checkpoint chỉ lưu ~33 M tensor trainable (≈130 MB) chứ không lưu 1.76 B tham số đóng
băng — quan trọng vì Colab hay đứt phiên và checkpoint phải ghi ra Drive thật nhanh.

**Bị đứt giữa chừng?** Chạy lại cell này với `--resume <đường dẫn checkpoint mới nhất>`.

In [ ]:
import subprocess

VRAM_GB = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]
).decode().split()[0]) / 1024

# Bảng 2.7 của VTON_PLAN.md
if VRAM_GB >= 70:
    BATCH_SIZE, ACCUM = 16, 1
elif VRAM_GB >= 38:
    BATCH_SIZE, ACCUM = 8, 2
else:
    BATCH_SIZE, ACCUM = 4, 4
    print("CẢNH BÁO: dưới 38 GB VRAM, batch nhỏ lại — sẽ chậm hơn đáng kể.")

print(f"VRAM {VRAM_GB:.0f} GB -> batch_size={BATCH_SIZE}, accumulation={ACCUM} "
      f"(effective {BATCH_SIZE * ACCUM})")

In [ ]:
RUN_DIR = "/content/drive/MyDrive/Fast_VTON_stage1"
MAX_STEPS = 40000

!mkdir -p {RUN_DIR}
!python scripts/train_vton_stage1.py \
    --cache outputs/vton_cache \
    --batch-size {BATCH_SIZE} \
    --gradient-accumulation-steps {ACCUM} \
    --max-steps {MAX_STEPS} \
    --output-dir {RUN_DIR} \
    --num-workers 2

In [ ]:
# Chạy lại sau khi Colab ngắt kết nối:
# latest = max(Path(RUN_DIR).glob("*.pt"), key=lambda p: p.stat().st_mtime)
# !python scripts/train_vton_stage1.py --cache outputs/vton_cache \
#     --batch-size {BATCH_SIZE} --gradient-accumulation-steps {ACCUM} \
#     --max-steps {MAX_STEPS} --output-dir {RUN_DIR} --resume {latest}

## 9. Kiểm tra G không bị đụng

Nguyên tắc cốt lõi của Stage 1: generator đóng băng. Chỉ `attn2.to_k/to_v` và `conv_in`
được phép đổi. Kiểm chứng bằng chính công cụ đã dùng để chứng minh tác giả đóng băng G.

In [ ]:
!python scripts/dissect_checkpoints.py --compare-generator

## 10. Đóng gói toàn bộ model thành 1 file → Google Drive

`export_bundle` ghi **một** file chứa mọi module cần cho inference cộng config để dựng
lại chúng: generator (đã mở 9 kênh) + garment encoder + mạng nghịch đảo + VAE + CLIP
vision + embedding rỗng. Máy đích **không cần** tải Hugging Face, **không cần**
`weights/`, chỉ cần file này.

Vì sao là bundle state-dict chứ không phải `torch.save(model)`: pickle một module sống sẽ
ghi lại đường dẫn class của từng submodule, nên file chết ngay khi có ai đổi tên class —
đúng thứ mà refactor làm. Bundle chỉ chứa tensor và config nên sống sót.

Kích thước fp16: khoảng **4.9 GB** (~2.46 B tham số).

In [ ]:
from pathlib import Path

import torch

from src.constants import INPAINTING_LATENT_CHANNELS
from src.models import AuxiliaryModel, InverseModel, IPSBV2Model
from src.vton import CheckpointConfig, GarmentEncoder, Stage1Config, Stage1Trainer

CHECKPOINTS = CheckpointConfig()
config = Stage1Config(output_dir=Path(RUN_DIR), max_steps=MAX_STEPS)

aux_model = AuxiliaryModel(device="cuda", load_text_encoder=False)
generator = IPSBV2Model(
    CHECKPOINTS.generator_dir,
    CHECKPOINTS.ip_adapter_path,
    aux_model,
    device="cuda",
    with_ip_mask_controller=True,
    inpainting_channels=INPAINTING_LATENT_CHANNELS,
)
garment_encoder = GarmentEncoder(CHECKPOINTS.garment_backbone).to("cuda")
inverse_model = InverseModel(CHECKPOINTS.inversion_dir, device="cuda", load_text_encoder=False)

# Nạp trọng số đã train vào bộ khung vừa dựng.
trainer = Stage1Trainer(generator, garment_encoder, config)
# final.pt < step_*.pt theo alphabet, nên chọn theo thời gian ghi.
trainer.load_checkpoint(max(Path(RUN_DIR).glob("*.pt"), key=lambda p: p.stat().st_mtime))
print("đã nạp checkpoint tại step", trainer.state.step)

In [ ]:
from src.vton import export_bundle

BUNDLE_PATH = "/content/drive/MyDrive/Fast_VTON_stage1/Fast_VTON_full.pt"

export_bundle(
    BUNDLE_PATH,
    generator=generator,
    garment_encoder=garment_encoder,
    inverse_model=inverse_model,
    step=trainer.state.step,
    height=config.data.height,
    width=config.data.width,
    include_frozen=True,        # tự chứa: máy đích không cần tải gì thêm
    dtype="fp16",
    null_embedding=torch.load("outputs/null_embedding.pt"),
)

In [ ]:
from src.vton import bundle_summary

summary = bundle_summary(BUNDLE_PATH)
print(f"file: {summary['file_size_bytes'] / 1e9:.2f} GB")
print(f"tổng tham số: {summary['total_parameters'] / 1e6:.1f} M\n")
for group, count in sorted(summary["parameters_by_group"].items()):
    print(f"  {group:18s} {count / 1e6:9.2f} M")
print("\nmanifest:")
for key, value in summary["manifest"].items():
    if not key.endswith("_config"):
        print(f"  {key:22s} {value}")

### Kiểm tra bundle nạp lại được

Chạy trên chính Colab để chắc file không hỏng trước khi mang về server 3090.

In [ ]:
import gc

# Giải phóng VRAM trước khi dựng lại từ bundle.
del generator, garment_encoder, inverse_model, aux_model, trainer
gc.collect()
torch.cuda.empty_cache()

from src.vton import load_bundle

bundle = load_bundle(BUNDLE_PATH, device="cuda")
print("step:", bundle.manifest.step)
print("độ phân giải:", bundle.manifest.height, "x", bundle.manifest.width)
print("kênh conv_in:", bundle.manifest.inpainting_channels)
print("có module đóng băng:", bundle.manifest.includes_frozen)
print("mạng nghịch đảo:", type(bundle.inversion_unet).__name__)
print("VAE:", type(bundle.vae).__name__)
print("\nBundle nạp lại thành công")

## Xong

`Fast_VTON_full.pt` đã nằm trên Drive. Trên server RTX 3090 chỉ cần:

```python
from src.vton import load_bundle

bundle = load_bundle("Fast_VTON_full.pt", device="cuda")
```

fp16 toàn bộ chiếm khoảng 3.2 GB VRAM — thoải mái trong 24 GB.

**Việc còn lại (Stage 2, xem mục 2.10 của `docs/VTON_PLAN.md`)**

- Mở F_θ (865.91 M), thêm DISTS + `L_regu` kiểu SDS theo Eq. 8 của paper.
- Bật `DataConfig.horizontal_flip` và cache cả hai chiều.
- **Ablation bỏ F_θ:** thay `inverted_noise` bằng nhiễu Gauss rồi đo lại. Nếu chất lượng
  tụt không đáng kể thì bỏ hẳn F_θ — inference còn một lượt UNet, nhanh gấp đôi. Đây là
  quyết định đáng giá nhất còn lại, và phải đo chứ không đoán.

In [ ]:
import google.colab
import time

# (Tùy chọn) Nghỉ 1 phút để đảm bảo các dữ liệu/model cuối cùng đã kịp lưu xong vào Google Drive
time.sleep(60) 

print("Quá trình train hoàn tất! Đang tiến hành ngắt phiên kết nối để tiết kiệm tài nguyên...")

In [ ]:
# Lệnh ngắt kết nối và giải phóng runtime ngay lập tức
google.colab.runtime.disconnect_all()